# Orthogonalized Sieve Empirical Likelihood

This notebook imports the implementation from `src/profile_sieve/` so notebook checks and package code stay in sync.

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from profile_sieve import (
    SimulatedDGP,
    LegendreSieve,
    SieveELEstimator,
    MonteCarloRunner,
    AcademicPlotter,
)

## Orthogonalization Check

In [3]:
T_sim = 500
K_sim = int(np.round(T_sim ** (1 / 3)))

dgp = SimulatedDGP(T_sim, rho=0.99, beta=0.0, phi=0.5, seed=42)
Y, X, X_lag, W_lag, _ = dgp.get_data()

basis = LegendreSieve(K_sim)
P_K = basis.construct(W_lag)
estimator = SieveELEstimator(basis)
w_c = estimator.step4_compute_orthogonalized_weights(X_lag, P_K)
ortho_check = np.sum(P_K.T * w_c, axis=1)

print("Verification of Step 4 Orthogonalization (sum P_K * w_c):")
print(ortho_check)
assert np.allclose(ortho_check, 0, atol=1e-10), "Orthogonalization mathematically failed!"
print("Orthogonalization structurally verified via Sieve weight projection.")

Verification of Step 4 Orthogonalization (sum P_K * w_c):
[-1.32227562e-13  8.10462808e-15  4.15778523e-14 -4.88498131e-15
 -7.42461648e-15  7.10542736e-15  1.41553436e-15 -1.12271303e-14]
Orthogonalization structurally verified via Sieve weight projection.


## Monte Carlo Smoke Run

In [4]:
# This quick check runs sequentially so it works reliably inside notebooks on Windows.
# Set RUN_FULL_PLOTS = True only when you want to write result/legacy/size_distortion.png
# and result/legacy/power_curve.png with a fresh run.
T_sim = 200
K_sim = int(np.round(T_sim ** (1 / 3)))
iterations_sim = 10

t0 = time.time()
runner = MonteCarloRunner(iterations=iterations_sim, T=T_sim, K=K_sim, n_jobs=1)
size = runner.simulate(rho=0.99, beta_true=0.0, beta_0=0.0, phi=0.5)
power = runner.simulate(rho=0.99, beta_true=0.1, beta_0=0.0, phi=0.5)

print(f"Size check at rho=0.99: EL={size[0]:.3f}, Score={size[1]:.3f}, OLS={size[2]:.3f}")
print(f"Power check at beta=0.1: EL={power[0]:.3f}, Score={power[1]:.3f}, OLS={power[2]:.3f}")
print(f"Smoke run elapsed: {time.time() - t0:.2f} seconds")

RUN_FULL_PLOTS = False
if RUN_FULL_PLOTS:
    AcademicPlotter.plot_size(runner)
    AcademicPlotter.plot_power(runner, rho=0.99)

Size check at rho=0.99: EL=0.300, Score=0.300, OLS=0.200
Power check at beta=0.1: EL=0.800, Score=0.700, OLS=0.900
Smoke run elapsed: 0.03 seconds
